# Introduction to JAX Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Setup and Data

We will train a 3-layer MLP on MNIST using JAX and Optax. 784 inputs, two hidden layers of 256 and 128 neurons, 10 output classes.

In [ ]:
```python

import jax

import jax.numpy as jnp

from jax import random

import optax

def get_mnist_data():

    from sklearn.datasets import fetch_openml

    mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')

    X = mnist.data.astype('float32') / 255.0

    y = mnist.target.astype('int')

    X_train, X_test = X[:60000], X[60000:]

    y_train, y_test = y[:60000], y[60000:]

    return X_train, y_train, X_test, y_test

In [ ]:
```

### Step 2: Initialize Parameters

No class. Just a function that returns a pytree:

In [ ]:
```python

def init_params(key):

    k1, k2, k3 = random.split(key, 3)

    scale1 = jnp.sqrt(2.0 / 784)

    scale2 = jnp.sqrt(2.0 / 256)

    scale3 = jnp.sqrt(2.0 / 128)

    params = {

        'layer1': {

            'w': scale1 * random.normal(k1, (784, 256)),

            'b': jnp.zeros(256),

        },

        'layer2': {

            'w': scale2 * random.normal(k2, (256, 128)),

            'b': jnp.zeros(128),

        },

        'layer3': {

            'w': scale3 * random.normal(k3, (128, 10)),

            'b': jnp.zeros(10),

        },

    }

    return params

In [ ]:
```

He-initialization, done manually. Three PRNG keys split from one seed. Every weight is an immutable array in a nested dict.

### Step 3: Forward Pass

In [ ]:
```python

def forward(params, x):

    x = jnp.dot(x, params['layer1']['w']) + params['layer1']['b']

    x = jax.nn.relu(x)

    x = jnp.dot(x, params['layer2']['w']) + params['layer2']['b']

    x = jax.nn.relu(x)

    x = jnp.dot(x, params['layer3']['w']) + params['layer3']['b']

    return x

def loss_fn(params, x, y):

    logits = forward(params, x)

    one_hot = jax.nn.one_hot(y, 10)

    return -jnp.mean(jnp.sum(jax.nn.log_softmax(logits) * one_hot, axis=-1))

In [ ]:
```

Pure functions. Params in, prediction out. No `self`, no stored state. `loss_fn` computes cross-entropy from scratch -- softmax, log, negative mean.

### Step 4: JIT-Compiled Training Step

In [ ]:
```python

@jax.jit

def train_step(params, opt_state, x, y):

    loss, grads = jax.value_and_grad(loss_fn)(params, x, y)

    updates, opt_state = optimizer.update(grads, opt_state, params)

    params = optax.apply_updates(params, updates)

    return params, opt_state, loss

@jax.jit

def accuracy(params, x, y):

    logits = forward(params, x)

    preds = jnp.argmax(logits, axis=-1)

    return jnp.mean(preds == y)

In [ ]:
```

`jax.value_and_grad` returns both the loss value and the gradients in one pass. The `@jax.jit` decorator compiles both functions to XLA. After the first call, each training step runs without touching Python.

### Step 5: Training Loop

In [ ]:
```python

optimizer = optax.adam(learning_rate=1e-3)

X_train, y_train, X_test, y_test = get_mnist_data()

X_train, X_test = jnp.array(X_train), jnp.array(X_test)

y_train, y_test = jnp.array(y_train), jnp.array(y_test)

key = random.PRNGKey(0)

params = init_params(key)

opt_state = optimizer.init(params)

batch_size = 128

n_epochs = 10

for epoch in range(n_epochs):

    key, subkey = random.split(key)

    perm = random.permutation(subkey, len(X_train))

    X_shuffled = X_train[perm]

    y_shuffled = y_train[perm]

    epoch_loss = 0.0

    n_batches = len(X_train) // batch_size

    for i in range(n_batches):

        start = i * batch_size

        xb = X_shuffled[start:start + batch_size]

        yb = y_shuffled[start:start + batch_size]

        params, opt_state, loss = train_step(params, opt_state, xb, yb)

        epoch_loss += loss

    train_acc = accuracy(params, X_train[:5000], y_train[:5000])

    test_acc = accuracy(params, X_test, y_test)

    print(f"Epoch {epoch + 1:2d} | Loss: {epoch_loss / n_batches:.4f} | "

          f"Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")

In [ ]:
```

10 epochs. ~97% test accuracy. The first epoch is slow (JIT compilation). Epochs 2-10 are fast.

Notice what is missing: no `.zero_grad()`, no `.backward()`, no `.step()`. The entire update is one composed function call. Gradients are computed, transformed by Adam, and applied to parameters -- all inside `train_step`.

## Exercises

In [ ]:
1. Add dropout to the MLP. In JAX, dropout requires a PRNG key -- thread a key through the forward pass and split it for each dropout layer. Compare test accuracy with and without.

2. Use `jax.vmap` to compute per-example gradients for a batch of 32 MNIST images. Compute the gradient norm for each example. Which examples have the largest gradients, and why?

3. Replace the manual forward function with a generic `mlp_forward(params, x)` that works for any number of layers. Use `jax.tree.leaves` to determine the depth automatically.

4. Benchmark the training step with and without `@jax.jit`. Time 100 steps of each. How large is the speedup on your hardware? What is the compilation overhead on the first call?

5. Implement gradient clipping by composing `optax.chain(optax.clip_by_global_norm(1.0), optax.adam(1e-3))`. Train with and without clipping. Plot the gradient norm over training to see the effect.